Multiagente — Predicción de Precios de Vehículos Eléctricos

Pipeline con 4 agentes:
1. **Normalizador** – limpieza, imputación y codificación
2. **Entrenador** – Random Forest Regressor
3. **Comunicador** – reporte de métricas
4. **Chatbot** – interfaz en lenguaje natural con Mistral AI


In [2]:
# Instalar dependencias (solo necesario en Colab)
import importlib, subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["pandas", "numpy", "scikit-learn", "requests"]:
    install(pkg)

print("✅ Dependencias listas.")


✅ Dependencias listas.


In [3]:
import pandas as pd
import numpy as np
import warnings
import requests

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings('ignore')
print("✅ Imports correctos.")


✅ Imports correctos.


## Dataset



In [4]:
from google.colab import files
import os

print("📂 Seleccioná tu archivo CSV:")
uploaded = files.upload()

# Tomar el nombre del archivo subido automáticamente
CSV_FILE = list(uploaded.keys())[0]
print(f"✅ Archivo '{CSV_FILE}' cargado correctamente.")


📂 Seleccioná tu archivo CSV:


Saving ev_market_2026.csv to ev_market_2026.csv
✅ Archivo 'ev_market_2026.csv' cargado correctamente.


In [5]:
df_ev = pd.read_csv(CSV_FILE)
print(f"✅ Dataset cargado. Dimensiones: {df_ev.shape}")
print("\nPrimeras filas:")
df_ev.head()


✅ Dataset cargado. Dimensiones: (2000, 24)

Primeras filas:


,brand,model,year,variant,price_usd,battery_capacity_kwh,range_miles,charging_speed_kw,acceleration_0_60_mph,top_speed_mph,...,body_type,cargo_volume_cubic_ft,weight_kg,safety_rating,autopilot_level,country_of_origin,market_segment,annual_sales_units,customer_rating,warranty_years
0,Volkswagen,ID. Buzz,2023,Performance,104880.80,118.7,400.0,234.5,3.04,195.0,...,Truck,54.2,2015.0,4,2,Germany,Luxury,202182,4.00,4
1,Toyota,bZ Compact SUV,2022,Premium,48217.41,58.8,219.0,148.1,5.77,159.0,...,SUV,69.0,1709.0,4,0,Japan,Mid-range,7146,3.56,3
2,GM/Chevrolet,Bolt EV,2024,Premium,49651.12,58.2,225.0,104.9,6.84,148.0,...,Van,77.0,1533.0,4,1,US,Mid-range,16590,3.70,3
3,Kia,Sportage EV,2024,Long Range,38131.56,102.5,349.0,66.5,4.66,176.0,...,SUV,65.8,1935.0,4,2,South Korea,Mid-range,127201,3.81,3
4,Tesla,Model 3,2022,Long Range,144079.87,93.9,314.0,298.5,5.64,165.0,...,SUV,38.3,2229.0,4,2,US,Luxury,196401,3.83,4


##Agente 1 — Normalizador

In [6]:
class NormalizerAgent:
    """Limpieza, imputación, codificación y escalado del dataset."""

    def __init__(self):
        self.scaler      = StandardScaler()
        self.num_imputer = SimpleImputer(strategy='mean')
        self.cat_imputer = SimpleImputer(strategy='most_frequent')

    def process(self, df, target_column):
        print("🤖 [Agente 1 - Normalizador]: Iniciando limpieza y normalización...")
        df_clean = df.copy()

        num_cols = df_clean.select_dtypes(include=['number']).columns.tolist()
        cat_cols = df_clean.select_dtypes(exclude=['number']).columns.tolist()

        if target_column in num_cols:
            num_cols.remove(target_column)
        if target_column in cat_cols:
            cat_cols.remove(target_column)

        # Imputación
        if num_cols:
            df_clean[num_cols] = self.num_imputer.fit_transform(df_clean[num_cols])
        if cat_cols:
            df_clean[cat_cols] = self.cat_imputer.fit_transform(df_clean[cat_cols])

        # One-Hot Encoding
        if cat_cols:
            df_clean = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

        df_clean = df_clean.astype(float)

        # Escalado
        if num_cols:
            df_clean[num_cols] = self.scaler.fit_transform(df_clean[num_cols])

        print("✅ [Agente 1 - Normalizador]: Dataset limpio, imputado, escalado y codificado.")
        return df_clean

TARGET_COL = 'price_usd'

agente_normalizador = NormalizerAgent()
df_limpio = agente_normalizador.process(df_ev, target_column=TARGET_COL)

print("\nVista rápida del dataset limpio:")
df_limpio.head(3)


🤖 [Agente 1 - Normalizador]: Iniciando limpieza y normalización...
✅ [Agente 1 - Normalizador]: Dataset limpio, imputado, escalado y codificado.

Vista rápida del dataset limpio:


,year,price_usd,battery_capacity_kwh,range_miles,charging_speed_kw,acceleration_0_60_mph,top_speed_mph,horsepower,torque_nm,seating_capacity,...,body_type_Truck,body_type_Van,country_of_origin_Germany,country_of_origin_Japan,country_of_origin_South Korea,country_of_origin_Sweden,country_of_origin_US,market_segment_Luxury,market_segment_Mid-range,market_segment_Premium
0,-0.729325,104880.80,2.193322,1.945833,0.979393,-1.827766,1.449530,1.251623,1.230640,1.199062,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-1.438440,48217.41,-0.790064,-0.603087,-0.087773,0.098326,-0.283235,-1.066225,-1.011165,-0.714844,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.020210,49651.12,-0.819948,-0.518592,-0.621357,0.853241,-0.812691,-0.291113,-0.357480,-0.714844,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


## Agente 2 — Entrenador

In [7]:
class TrainerAgent:
    """Entrena un Random Forest Regressor y calcula métricas."""

    def __init__(self):
        self.model = RandomForestRegressor(
            n_estimators=200, max_depth=20, random_state=42
        )

    def process(self, df, target_column):
        print("🤖 [Agente 2 - Entrenador]: Iniciando separación y entrenamiento...")

        X = df.drop(columns=[target_column])
        y = df[target_column]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        print(f"   Entrenamiento: {X_train.shape[0]} muestras | Prueba: {X_test.shape[0]} muestras")
        print("   ⏳ Entrenando Random Forest (puede tardar ~30 s)...")

        self.model.fit(X_train, y_train)
        predictions = self.model.predict(X_test)

        metrics = {
            'MSE' : mean_squared_error(y_test, predictions),
            'RMSE': np.sqrt(mean_squared_error(y_test, predictions)),
            'R2'  : r2_score(y_test, predictions),
        }

        print("✅ [Agente 2 - Entrenador]: Modelo entrenado.")
        return self.model, metrics

agente_entrenador = TrainerAgent()
modelo_entrenado, metricas_modelo = agente_entrenador.process(df_limpio, target_column=TARGET_COL)

print(f"\n   R²   = {metricas_modelo['R2']:.4f}")
print(f"   RMSE = ${metricas_modelo['RMSE']:,.2f} USD")


🤖 [Agente 2 - Entrenador]: Iniciando separación y entrenamiento...
   Entrenamiento: 1600 muestras | Prueba: 400 muestras
   ⏳ Entrenando Random Forest (puede tardar ~30 s)...
✅ [Agente 2 - Entrenador]: Modelo entrenado.

   R²   = 0.9538
   RMSE = $7,257.65 USD


##  Agente 3 — Comunicador

In [8]:
class CommunicatorAgent:
    """Genera el reporte final con las métricas del modelo."""

    def process(self, metrics):
        print("\n🤖 [Agente 3 - Comunicador]: Generando reporte final...\n")

        reporte  = "📊 **REPORTE DE RESULTADOS DEL MODELO PREDICTIVO DE PRECIOS EV** 📊\n"
        reporte += "-" * 65 + "\n"
        reporte += "El proceso de entrenamiento ha finalizado con éxito.\n\n"
        reporte += "📉 **Tipo de Tarea:** Regresión (Predicción de 'price_usd')\n"
        reporte += f"🎯 **R² Score (Precisión general):** {metrics['R2'] * 100:.2f}%\n"
        reporte += f"📏 **Margen de Error Promedio (RMSE):** ${metrics['RMSE']:,.2f} USD\n"
        reporte += "\n💡 *Interpretación:*\n"
        reporte += (
            f"El modelo explica el {metrics['R2'] * 100:.2f}% de la variación en los precios.\n"
            f"En promedio, las predicciones difieren ${metrics['RMSE']:,.2f} USD del precio real.\n"
        )
        reporte += "\n🚀 El pipeline multiagente ha concluido correctamente.\n"

        print(reporte)
        return reporte

agente_comunicador = CommunicatorAgent()
reporte_final = agente_comunicador.process(metricas_modelo)



🤖 [Agente 3 - Comunicador]: Generando reporte final...

📊 **REPORTE DE RESULTADOS DEL MODELO PREDICTIVO DE PRECIOS EV** 📊
-----------------------------------------------------------------
El proceso de entrenamiento ha finalizado con éxito.

📉 **Tipo de Tarea:** Regresión (Predicción de 'price_usd')
🎯 **R² Score (Precisión general):** 95.38%
📏 **Margen de Error Promedio (RMSE):** $7,257.65 USD

💡 *Interpretación:*
El modelo explica el 95.38% de la variación en los precios.
En promedio, las predicciones difieren $7,257.65 USD del precio real.

🚀 El pipeline multiagente ha concluido correctamente.



## Predictor Interactivo de Precios

Ingresá las características de un vehículo y el modelo estimará su precio.


🤖 [Agente Predictor]: Ingresá las características del vehículo.
Escribí 'salir' en la marca para terminar.

----------------------------------------
1. Marca (ej. Tesla, Kia, Ford): salir
🤖 [Agente Predictor]: ¡Nos vemos!


##  Agente 4 — Chatbot Analista (Mistral AI)



🤖 [Agente 4 - Chatbot Analista con Mistral AI]
¡Hola! Preguntame sobre los resultados o los vehículos. Escribí 'salir' para terminar.

👤 Vos: Quiero una novia{
   ⏳ Pensando...

🤖 Mistral: ¡Ah, entiendo! Pero mi especialidad es analizar datos de vehículos eléctricos. Si tienes preguntas sobre modelos, autonomía, precios, comparativas o tendencias del mercado de autos eléctricos, estaré encantado de ayudarte con datos precisos y análisis basados en el modelo que mencioné.

Si buscas consejos sobre relaciones o citas, te recomendaría consultar a un experto en el tema. 😊 ¿En qué puedo asistirte con los vehículos eléctricos?

👤 Vos: mmm
   ⏳ Pensando...

🤖 Mistral: ¡Hola! Soy tu asistente experto en análisis de datos de vehículos eléctricos. ¿En qué puedo ayudarte hoy? 😊

Por ejemplo, puedo:
- **Predecir precios** de EVs con alta precisión (R²: 95.38%, RMSE: $7,257.65).
- **Analizar tendencias** por marca, autonomía o año.
- **Recomendar modelos** según tu presupuesto o necesidades.

Dime

KeyboardInterrupt: Interrupted by user